In [11]:
# Import Libraries

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_score
)

from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import (
    RandomForestRegressor,
    StackingRegressor
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

print("Libraries imported successfully!")

Libraries imported successfully!


In [12]:
# Summary
# Load the preprocessed dataset and
# split it into training and validation sets.

X = pd.read_csv(
    "../Outputs/pipeline1_train_preprocessed.csv"
)

y = np.log1p(
    pd.read_csv(
        "../Outputs/pipeline1_train_target.csv"
    )["SalePrice"]
)

print("Dataset loaded successfully!\n")

print(f"Dataset Shape : {X.shape}")

print(f"Missing Values: {X.isnull().sum().sum()}")

Dataset loaded successfully!

Dataset Shape : (1460, 208)
Missing Values: 1468


In [13]:
X_train, X_test, y_train, y_test = train_test_split(

    X.fillna(0),

    y,

    test_size=0.2,

    random_state=42

)

# Create 5-Fold Cross Validation

kf = KFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

print("Train/Test Split completed!\n")

print(f"Train Shape : {X_train.shape}")

print(f"Test Shape  : {X_test.shape}")

Train/Test Split completed!

Train Shape : (1168, 208)
Test Shape  : (292, 208)


In [14]:
# RMSE Cross Validation Function

def rmse_cv(

    model,

    X,

    y

):

    scores = cross_val_score(

        model,

        X,

        y,

        scoring="neg_root_mean_squared_error",

        cv=kf,

        n_jobs=-1

    )

    return -scores

In [15]:
# I Create tuned machine learning models.

xgb = XGBRegressor(

    colsample_bytree=0.8,

    learning_rate=0.03,

    max_depth=3,

    n_estimators=1000,

    subsample=0.8,

    reg_alpha=0.1,

    reg_lambda=1.0,

    random_state=42,

    n_jobs=-1,

    verbosity=0

)

lgbm = LGBMRegressor(

    learning_rate=0.03,

    max_depth=3,

    n_estimators=1000,

    num_leaves=15,

    subsample=0.7,

    random_state=42,

    n_jobs=-1,

    verbosity=-1

)

rf = RandomForestRegressor(

    n_estimators=700,

    max_depth=20,

    max_features="sqrt",

    min_samples_leaf=1,

    min_samples_split=2,

    random_state=42,

    n_jobs=-1

)

print("Models created successfully!")

xgb.fit(
    X_train,
    y_train
)

lgbm.fit(
    X_train,
    y_train
)

rf.fit(
    X_train,
    y_train
)

print("All models trained successfully!")

Models created successfully!
All models trained successfully!


In [ ]:
# Cross Validation Comparison

results = {}

models = {

    "XGBoost": xgb,

    "LightGBM": lgbm,

    "Random Forest": rf

}

for name, model in models.items():

    scores = rmse_cv(

        model,

        X.fillna(0),

        y

    )

    results[name] = [

        scores.mean(),

        scores.std()

    ]

results = pd.DataFrame(

    results,

    index=[

        "Mean RMSE",

        "Std RMSE"

    ]

).T

results = results.sort_values(

    by="Mean RMSE"

)

display(results)

,Mean RMSE,Std RMSE
XGBoost,0.126011,0.017372
LightGBM,0.128604,0.015132
Random Forest,0.136866,0.015931


In [17]:
df1 = pd.read_csv("../Outputs/pipeline1_train_preprocessed.csv")
df2 = pd.read_csv("../Outputs/pipeline1_train_feature_engineered.csv")

print(df1.shape)
print(df2.shape)

print(df1.columns.equals(df2.columns))

(1460, 208)
(1460, 93)
False


In [18]:
print(xgb.get_params())

{'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': True, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.03, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 3, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 1000, 'n_jobs': -1, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.8, 'tree_method': None, 'validate_parameters': None, 'verbosity': 0}


In [19]:
scores = rmse_cv(
    xgb,
    X.fillna(0),
    y
)

print(scores)
print(scores.mean())
print(scores.std())

[0.13096101 0.11242454 0.157232   0.12063661 0.10879959]
0.12601074906650964
0.017372176391946888


In [20]:
print(X.shape)
print(X.fillna(0).isnull().sum().sum())

(1460, 208)
0
